Fish to Fish Regression Test Errors by Species

In [2]:

import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


df = pd.read_csv(
    "../adore_dataset/processed/t-F2F_mortality.csv",
    dtype=str
)

#species columns 
df["Full_Species"] = (
    df["tax_genus"].astype(str)
    + " "
    + df["tax_species"].astype(str)
)

species_col = "Full_Species"

# features
target = "result_conc1_mean_mol_log"

categorical_cols = [
    "test_exposure_type",
    "result_conc1_type",
    "test_media_type"
]

ordinal_cols = ["result_obs_duration_mean"]

numeric_cols = [
    "chem_mw",
    "chem_ws",
    "chem_mp",
    "chem_rdkit_clogp"
]

# numeric columns
for col in numeric_cols + ordinal_cols + [target]:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

# macc fingerprint
def parse_maccs(value):
    if not isinstance(value, str):
        return None

    value = value.strip()

    if len(value) != 167:
        return None

    if not set(value).issubset({"0", "1"}):
        return None

    return [int(bit) for bit in value]

valid_fp = df["chem_MACCS_fp"].apply(parse_maccs)

df = df.loc[valid_fp.notna()].copy()

maccs = pd.DataFrame(
    valid_fp.loc[df.index].tolist(),
    index=df.index,
    columns=[f"MACCS_{i:03d}" for i in range(167)]
)

df.value_counts()

test_id  reference_number  test_cas   test_location  test_exposure_type  test_control_type  test_media_type  test_application_freq_unit  test_organism_lifestage  result_id  result_effect  result_endpoint  result_obs_duration_mean  result_conc1_type  result_conc1_mean_op  result_conc1_mean  result_conc1_mean_mol  media_ph_mean  media_temperature_mean  tax_all                                                      tax_name         tax_class       tax_order      tax_family   tax_genus   tax_species  tax_gs               species_number  tax_group  tax_pdm_available  tax_eco_climate  tax_eco_ecozone  tax_eco_food  tax_eco_migrate5  tax_eco_migrate2  tax_ps_ampv  tax_ps_ampkap  tax_ps_amppm  tax_lh_amd  tax_lh_lbcm  tax_lh_lpcm  tax_lh_licm  tax_lh_ri#/d  result_conc1_mean_binary  result_conc1_mean_log  result_conc1_mean_mol_log  chem_dtxsid    chem_name                      test_cas_name                             chem_sf      chem_mw     chem_mp    chem_ws   chem_ws_binary  chem_dtxcid     

In [3]:
# X and y
X = pd.concat([
    df[categorical_cols + ordinal_cols + numeric_cols],
    maccs
], axis=1)

y = df[target]

# test splits
train = df["split_occurrence"] != "test"
test = df["split_occurrence"] == "test"

valid = y.notna() & df[species_col].notna()

train = train & y.notna()
test = test & valid

X_train = X.loc[train]
X_test = X.loc[test]

y_train = y.loc[train]
y_test = y.loc[test]

In [4]:
# maccs bits
fp_cols = [
    c for c in maccs.columns
    if X_train[c].std() >= 0.1
]

features = (
    categorical_cols
    + ordinal_cols
    + numeric_cols
    + fp_cols
)

X_train = X_train[features]
X_test = X_test[features]

In [5]:
# preprocessing
numeric_features = (
    ordinal_cols + numeric_cols + fp_cols
)

preprocessing = ColumnTransformer([
    (
        "numeric",
        make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ),
        numeric_features
    ),
    (
        "categorical",
        make_pipeline(
            SimpleImputer(
                strategy="most_frequent"
            ),
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
        categorical_cols
    )
])


In [6]:
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor)
from sklearn.base import clone
from sklearn.metrics import (mean_absolute_error,root_mean_squared_error,r2_score)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}



In [ ]:
# grid search
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

# use predefined split to define folds
fold_ids = pd.to_numeric(df.loc[X_train.index, "split_occurrence"], errors="coerce")
assert fold_ids.notna().all(), "non-numeric fold ids — check split_occurrence values"
cv = PredefinedSplit(fold_ids.astype(int).values)

# define grids to search
param_grids = {
    "Ridge":       {"reg__alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso":       {"reg__alpha": [1e-4, 1e-3, 1e-2, 0.1]},
    "Elastic Net": {"reg__alpha": [1e-3, 1e-2, 0.1],
                    "reg__l1_ratio": [0.2, 0.5, 0.8]},
    "KNN":         {"reg__n_neighbors": [3, 5, 7, 11, 15],
                    "reg__weights": ["uniform", "distance"]},
    "SVR":         {"reg__C": [0.3, 1, 3, 10, 30],
                    "reg__epsilon": [0.05, 0.1, 0.2, 0.5],
                    "reg__gamma": ["scale", 0.01, 0.03]},
    "Decision Tree": {"reg__max_depth": [5, 10, 20, None],
                      "reg__min_samples_leaf": [1, 5, 10, 20]},
    "Random Forest": {"reg__n_estimators": [300],
                      "reg__max_depth": [10, 20, None],
                      "reg__max_features": ["sqrt", 0.3, 0.6],
                      "reg__min_samples_leaf": [1, 3]},
    "Gradient Boosting": {"reg__n_estimators": [200, 500],
                          "reg__learning_rate": [0.03, 0.1],
                          "reg__max_depth": [2, 3, 4],
                          "reg__subsample": [0.8, 1.0]},
    # Linear Regression has nothing to tune
}

# ---- run the searches ----
searches = {}
tuning_summary = []

for name, regressor in models.items():
    pipe = Pipeline([
        ("prep", clone(preprocessing)), 
        ("reg", clone(regressor))])
    grid = param_grids.get(name)

    if not grid:                       # e.g. Linear Regression
        searches[name] = pipe.fit(X_train, y_train)
        continue

    gs = GridSearchCV(
        pipe, grid,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        refit=True,
        verbose=1,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs.best_estimator_   # already refit on all training folds

    tuning_summary.append({
        "Model": name,
        "CV_RMSE": -gs.best_score_,
        "Best_Params": gs.best_params_,
    })
    print(f"{name:18s} CV RMSE {-gs.best_score_:.3f}  {gs.best_params_}")

tuning_summary = pd.DataFrame(tuning_summary).sort_values("CV_RMSE")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Ridge              CV RMSE 1.150  {'reg__alpha': 100}
Fitting 5 folds for each of 4 candidates, totalling 20 fits


/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.386329e+02, tolerance: 4.741e+00
  model = cd_fast.enet_coordinate_descent(
/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.219387e+01, tolerance: 4.423e+00
  model = cd_fast.enet_coordinate_descent(
/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the

Lasso              CV RMSE 1.111  {'reg__alpha': 0.01}
Fitting 5 folds for each of 9 candidates, totalling 45 fits


/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.162373e+01, tolerance: 4.741e+00
  model = cd_fast.enet_coordinate_descent(
/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.172558e+01, tolerance: 4.423e+00
  model = cd_fast.enet_coordinate_descent(
/Users/chelsygailpena/SourceCodes/Fish_Toxicity/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the

Elastic Net        CV RMSE 1.105  {'reg__alpha': 0.1, 'reg__l1_ratio': 0.2}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
KNN                CV RMSE 1.208  {'reg__n_neighbors': 15, 'reg__weights': 'distance'}
Fitting 5 folds for each of 60 candidates, totalling 300 fits


In [7]:
# train and error calculation
model_results = []
all_predictions = []

for name, regressor in models.items():

    pipeline = make_pipeline(
        clone(preprocessing),
        clone(regressor)
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    model_results.append({
        "Model": name,
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    predictions = pd.DataFrame({
        "Species": df.loc[X_test.index, species_col].values,
        "Actual": y_test.values,
        "Predicted": y_pred,
        "Model": name
    })

    predictions["Absolute_Error"] = abs(
        predictions["Actual"] - predictions["Predicted"]
    )

    all_predictions.append(predictions)

In [8]:
# comparing models
model_comparison = pd.DataFrame(
    model_results
).sort_values("RMSE")

print("\nMODEL COMPARISON")
print(model_comparison.to_string(index=False))

# all predictions concatonaed
all_errors = pd.concat(
    all_predictions, ignore_index=True
)

# error by species and model

species_model_errors = (
    all_errors.groupby(["Model", "Species"])
    .agg(
        MAE= ("Absolute_Error", "mean"),
        RMSE=(
            "Absolute_Error",
            lambda x: np.sqrt(np.mean(x**2))
        ),
        Count=("Absolute_Error", "size")
    )
    .reset_index()
)


MODEL COMPARISON
            Model     RMSE      MAE       R2
              SVR 0.986290 0.723425 0.684615
    Random Forest 1.040736 0.775941 0.648833
Gradient Boosting 1.051203 0.827325 0.641734
              KNN 1.119618 0.808318 0.593583
Linear Regression 1.134927 0.917361 0.582393
            Ridge 1.134931 0.917339 0.582390
    Decision Tree 1.342713 0.982221 0.415481
      Elastic Net 1.475269 1.198285 0.294374
            Lasso 1.688802 1.369309 0.075325


In [9]:
# top 10 highest errors
for name in models:
    print(f"\n{name} - Highest Error Species")

    top = (
        species_model_errors[
            species_model_errors["Model"] == name
        ]
        .sort_values("MAE", ascending=False)
        .head(10)
    )

    print(top.to_string(index=False))

#compare spceies
species_summary = (
    species_model_errors.groupby("Species")
    .agg(
        Average_MAE=("MAE", "mean"),
        Average_RMSE=("RMSE", "mean"),
        Count=("Count", "first")
    )
    .sort_values("Average_MAE", ascending=False)
)

print("\nSPECIES WITH GREATEST AVERAGE ERROR")
print(species_summary.head(15).to_string())





Linear Regression - Highest Error Species
            Model                Species      MAE     RMSE  Count
Linear Regression           Gadus morhua 2.757201 2.758106      2
Linear Regression        Clupea harengus 2.025726 2.034095      4
Linear Regression       Monopterus albus 1.900071 1.900082      4
Linear Regression            Esox lucius 1.889307 2.086975      9
Linear Regression      Leuresthes tenuis 1.871054 1.883200     26
Linear Regression       Fundulus grandis 1.829416 1.829416      1
Linear Regression Nothobranchius furzeri 1.748734 1.751178      8
Linear Regression     Gambusia holbrooki 1.633214 1.633265      2
Linear Regression     Lagodon rhomboides 1.622679 1.795703      4
Linear Regression          Sparus aurata 1.615090 1.690368      2

Ridge - Highest Error Species
Model                Species      MAE     RMSE  Count
Ridge           Gadus morhua 2.756712 2.757617      2
Ridge        Clupea harengus 2.024641 2.033016      4
Ridge       Monopterus albus 1.900990 

In [10]:
# save errors for report in csv
model_comparison.to_csv(
    "model_comparison.csv", index=False
)

species_model_errors.to_csv(
    "species_model_errors.csv", index=False
)

species_summary.to_csv(
    "species_error_summary.csv"
)


In [13]:
fish_top10_filtered = (
    species_summary[species_summary["Count"] >= 5]
    .sort_values("Average_MAE", ascending=False)
    .head(10)
)

print(fish_top10_filtered)

                             Average_MAE  Average_RMSE  Count
Species                                                      
Leuresthes tenuis               1.980293      2.006182     26
Esox lucius                     1.758149      2.016684      9
Menidia menidia                 1.531477      1.716167     42
Channa punctata                 1.518667      2.097916    100
Nothobranchius furzeri          1.513124      1.542869      8
Carassius carassius             1.346164      1.633262     13
Hypophthalmichthys molitrix     1.283140      1.355881     14
Clarias gariepinus              1.207698      1.545716     17
Morone saxatilis                1.158355      1.396896     38
Tinca tinca                     1.137565      1.301926      5
